# Attention encoder validation by layer count
Evaluate 10,000 randomly selected spectra per layer count (1?10), without replacement. Apply fixed Poisson noise to encoder inputs; compare frozen LRN_AD reconstructions with clean expected spectra. Models remain frozen and checkpoint files are read only.

This is a stratified dataset evaluation, not a new independent held-out dataset. Training overlap is reported explicitly. Four examples per row are selected nearest the p25, p50, p75 and p95 per-spectrum reconstruction losses.

In [ ]:
from pathlib import Path
import sys, json, hashlib, struct
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
try:
    import tomllib
except ImportError:
    import tomli as tomllib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'ibeamlab/lrn_ad.py').exists())
sys.path.insert(0, str(ROOT))
import ibeamlab as ibl
from ibeamlab import LRN_AD
from ibeamlab.spectrum_encoder import SpectrumAttentionEncoder
from ibeamlab._native import native
EXAMPLES = ROOT/'examples'

## 1. Checkpoints and evaluation settings

In [ ]:
ENCODER_CHECKPOINT = EXAMPLES/'artifacts/encoder_attention_lrn_ad/encoder_best.pt'
LRN_AD_CHECKPOINT = EXAMPLES/'artifacts/lrn_ad_rbs_10layers/lrn_ad_state.pt'
CONFIG_PATH = EXAMPLES/'multilayer-generation.toml'
SAMPLES_PER_LAYER = 10_000
LAYER_COUNTS = tuple(range(1, 11))
SEED = 20261010
BATCH_SIZE = 512
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(8)
REPORT_DIR = EXAMPLES/'artifacts/encoder_attention_lrn_ad/validation'

def file_hash(path):
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

checkpoint = torch.load(ENCODER_CHECKPOINT, map_location='cpu', weights_only=True)
signature = checkpoint['signature']
if signature['model_type'] != 'SpectrumAttentionEncoder':
    raise ValueError('Choose an attention encoder checkpoint')
for path, key in ((CONFIG_PATH, 'config_sha256'), (LRN_AD_CHECKPOINT, 'decoder_sha256')):
    if file_hash(path) != signature[key]:
        raise ValueError(f'Checkpoint/config differs from encoder training: {path}')
lrn_checkpoint = torch.load(LRN_AD_CHECKPOINT, map_location='cpu', weights_only=False)
configuration = ibl.load_generation_configuration(CONFIG_PATH)
MAX_LAYERS = signature['max_layers']
ELEMENTS = tuple(signature['elements'])
METHOD = lrn_checkpoint['method']
WIDTH = signature['input_size']
OUTPUT_SCALE = float(signature['output_scale'])
AD_FACTOR = float(signature['decoder_input_factor'])
if max(LAYER_COUNTS) > MAX_LAYERS or SAMPLES_PER_LAYER < 4:
    raise ValueError('Invalid layer counts/sample count')
study = configuration.study(MAX_LAYERS)
device = torch.device(DEVICE)
model = SpectrumAttentionEncoder(WIDTH, signature['output_size'], **signature['architecture'])
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device).eval().requires_grad_(False)
lrn = LRN_AD(study, {METHOD: WIDTH}, elements=ELEMENTS,
    ad_input_factor=AD_FACTOR, **lrn_checkpoint['architecture'])
lrn.load_state_dict(lrn_checkpoint['model_state_dict'])
lrn.to(device).eval().requires_grad_(False)
weights = torch.ones(WIDTH, device=device)
weights[signature['tail_start']:] = signature['tail_weight']
weights /= weights.mean()
print(f"Loaded protected best epoch {checkpoint['best_epoch']}; saved validation reconstruction {checkpoint['best_validation_reconstruction']:.6g}")
print(f'Evaluation device: {device}; {SAMPLES_PER_LAYER:,} samples ? {len(LAYER_COUNTS)} layer counts')

## 2. Stream stratified samples and reconstruct
Only selected records are retained in memory, one layer count at a time. Dataset shard format/header and generation provenance are checked. Sampling uses the same sorted-shard record order as training.

The primary metric is exactly the training loss, computed **per spectrum** before averaging:

$$\chi_i^2=\frac1C\sum_c w_c\frac{(s\hat y_{ic}-s y_{ic})^2}{s y_{ic}+1},\qquad s=0.01.$$

Channel weights are 5 above channel 1100 and 1 below, normalized to mean 1. The scale and weights are read from the checkpoint. This is a regularized, weighted chi?-like loss, not a statistical reduced chi?. True-AD forward reconstruction is reported alongside the encoder result as a reference, not a strict lower bound.

In [ ]:
def template_table(text):
    value = tomllib.loads(text)
    value.pop('format', None); value.pop('format_version', None)
    return value

def dataset_info(path, layers):
    ds = ibl.open_dataset(path)
    if not ds.metadata.complete or ds.metadata.format_version != 1 or not len(ds):
        raise ValueError(f'Incomplete/unsupported dataset: {path}')
    case = configuration.study(layers)
    cfg = case._config()
    provenance = tomllib.loads(ds._reader.metadata.generation_config_toml)
    if (provenance.get('setup') != template_table(native.sample.setup_to_toml(cfg.setup))
            or provenance.get('sample') != template_table(native.sample.sample_to_toml(cfg.sample))):
        raise ValueError(f'Dataset setup/sample differs: {path}')
    methods = [dict(label=m.label, iba_method=m.iba_method,
                    reference_file=Path(m.reference_file).as_posix()) for m in cfg.methods]
    if provenance.get('methods') != methods or set(ds.metadata.parameter_names) != set(case.parameter_names):
        raise ValueError(f'Dataset method/parameter schema differs: {path}')
    stored_targets = {p['name']: p['target'] for p in provenance.get('parameters', [])}
    for parameter in case.parameters:
        expected = (dict(type='layer_thickness', layer=parameter.layer) if parameter.kind == 'layer_thickness'
                    else dict(type='species_concentration', layer=parameter.layer, element=parameter.element))
        if stored_targets.get(parameter.name) != expected:
            raise ValueError(f'Dataset parameter target differs: {path}')
    return ds, case

def stream_selected(path, wanted):
    # IBEAMDS1 native little-endian shards; avoid Dataset.records/read_all.
    local = 0
    for shard in sorted(path.glob('part-*.ibd')):
        if not wanted: break
        with shard.open('rb') as f:
            def exact(n):
                data = f.read(n)
                if len(data) != n: raise EOFError(f'Truncated shard: {shard}')
                return data
            def number(fmt): return struct.unpack(fmt, exact(struct.calcsize(fmt)))[0]
            def string():
                size = number('<I')
                if size > 16*1024*1024: raise ValueError('Invalid string size')
                return exact(size).decode()
            if exact(8) != b'IBEAMDS1': raise ValueError(f'Invalid shard header: {shard}')
            end = shard.stat().st_size
            while f.tell() < end and wanted:
                number('<Q'); string()
                n = number('<I'); keep = local in wanted
                parameters = np.frombuffer(exact(n*8), dtype='<f8').copy() if keep else None
                if not keep: f.seek(n*8, 1)
                counts = None
                for _ in range(number('<I')):
                    label = string(); channels = number('<Q')
                    if keep and label == METHOD:
                        counts = np.frombuffer(exact(channels*4), dtype='<f4').copy()
                    else: f.seek(channels*4, 1)
                if f.tell() > end: raise EOFError(f'Truncated shard: {shard}')
                if keep:
                    if counts is None: raise ValueError(f'Missing {METHOD}: {shard}')
                    yield wanted.pop(local), parameters, counts
                local += 1
    if wanted: raise ValueError(f'Selected records missing: {path}')

def true_density(parameters, names, case):
    columns = dict(zip(names, parameters))
    ad = np.zeros((MAX_LAYERS, len(ELEMENTS)), dtype=np.float32)
    for i, layer in enumerate(case.experiment.sample.layers):
        thickness = layer.thickness
        fractions = np.array([next(s.concentration for s in layer.species if s.element == e)
                              for e in ELEMENTS], dtype=np.float32)
        for p in case.parameters:
            if p.layer != i: continue
            if p.kind == 'layer_thickness': thickness = columns[p.name]
            elif p.kind == 'concentration': fractions[ELEMENTS.index(p.element)] = columns[p.name]
        ad[i] = thickness*fractions
    return ad.reshape(-1)*AD_FACTOR

def per_sample_loss(predicted_scaled, clean):
    target = clean*OUTPUT_SCALE
    return (weights*(predicted_scaled-target).square()/(target+1)).mean(1)

def summarize(values):
    return dict(samples=len(values), mean=float(np.mean(values, dtype=np.float64)),
        **dict(zip(('median','p90','p95','p99','max'),
                   map(float, np.quantile(values, (.5,.9,.95,.99,1))))))

manifest = signature['dataset_manifest']
entries = []
offset = 0
for entry in manifest:
    path = Path(entry['path'])
    layers = int(path.name.split('_')[-1])
    ds, case = dataset_info(path, layers)
    current = [(f.name, f.stat().st_size, f.stat().st_mtime_ns)
               for f in sorted(path.iterdir()) if f.is_file()]
    if current != [tuple(item) for item in entry['files']]:
        raise ValueError(f'Dataset changed since training: {path}')
    entries.append((path, layers, ds, case, offset))
    offset += len(ds)
train_rows = checkpoint['train_indices'].numpy()
results, previews, loss_arrays, reference_arrays, selected_rows = {}, {}, {}, {}, {}
for layers in tqdm(LAYER_COUNTS, desc='Layer counts'):
    cases = [entry for entry in entries if entry[1] == layers]
    total = sum(len(entry[2]) for entry in cases)
    if total < SAMPLES_PER_LAYER:
        raise ValueError(f'{layers} layers: need {SAMPLES_PER_LAYER:,}, only {total:,} available')
    choices = np.sort(np.random.default_rng(SEED+layers).choice(total, SAMPLES_PER_LAYER, replace=False))
    clean = np.zeros((SAMPLES_PER_LAYER, WIDTH), np.float32)
    true_ad = np.zeros((SAMPLES_PER_LAYER, signature['output_size']), np.float32)
    global_rows = np.empty(SAMPLES_PER_LAYER, np.int64)
    group_offset = 0
    for path, _, ds, case, global_offset in cases:
        positions = np.flatnonzero((choices >= group_offset) & (choices < group_offset+len(ds)))
        wanted = {int(choices[j]-group_offset): int(j) for j in positions}
        for j, parameters, counts in stream_selected(path, wanted):
            if len(counts) > WIDTH or len(parameters) != len(ds.metadata.parameter_names):
                raise ValueError(f'Record dimensions differ: {path}')
            clean[j, :len(counts)] = counts
            true_ad[j] = true_density(parameters, ds.metadata.parameter_names, case)
            global_rows[j] = global_offset+choices[j]-group_offset
        group_offset += len(ds)
    if not np.isfinite(clean).all() or (clean < 0).any() or not np.isfinite(true_ad).all():
        raise ValueError('Invalid sampled spectra/AD')
    noisy = np.empty_like(clean)
    generator = torch.Generator().manual_seed(SEED+100+layers)
    losses, references = [], []
    with torch.inference_mode():
        for start in range(0, len(clean), BATCH_SIZE):
            stop = min(start+BATCH_SIZE, len(clean))
            noisy[start:stop] = torch.poisson(torch.from_numpy(clean[start:stop]), generator=generator).numpy()
            target = torch.from_numpy(clean[start:stop]).to(device)
            prediction = lrn(model(torch.from_numpy(noisy[start:stop]).to(device)))
            reference = lrn(torch.from_numpy(true_ad[start:stop]).to(device))
            losses.append(per_sample_loss(prediction, target).cpu().numpy())
            references.append(per_sample_loss(reference, target).cpu().numpy())
    losses, references = np.concatenate(losses), np.concatenate(references)
    if not np.isfinite(losses).all() or not np.isfinite(references).all():
        raise FloatingPointError('Nonfinite reconstruction loss')
    selected_rows[layers] = global_rows
    loss_arrays[layers], reference_arrays[layers] = losses, references
    overlap = int(np.isin(global_rows, train_rows).sum())
    results[layers] = dict(encoder=summarize(losses), true_ad=summarize(references), training_overlap=overlap)
    chosen = []
    for q in (.25,.5,.75,.95):
        order = np.argsort(np.abs(losses-np.quantile(losses,q)))
        chosen.append(next(int(i) for i in order if i not in chosen))
    with torch.inference_mode():
        reconstructed = lrn(model(torch.from_numpy(noisy[chosen]).to(device))).cpu().numpy()/OUTPUT_SCALE
    previews[layers] = dict(clean=clean[chosen], noisy=noisy[chosen], reconstructed=reconstructed,
                            losses=losses[chosen], indices=global_rows[chosen])
    print(f'{layers:2d} layers: mean={losses.mean():.6g}; median={np.median(losses):.6g}; train overlap={overlap:,}/{len(losses):,}')
    del clean, true_ad, noisy

## 3. Reconstruction statistics and saved report

In [ ]:
print('Encoder reconstruction chi? (training definition)')
print(f"{'Layers':>6} {'N':>8} {'Mean':>12} {'Median':>12} {'p90':>12} {'p95':>12} {'p99':>12} {'Max':>12}")
for layers in LAYER_COUNTS:
    r = results[layers]['encoder']
    print(f"{layers:6d} {r['samples']:8,d}" + ''.join(f" {r[key]:12.6g}" for key in ('mean','median','p90','p95','p99','max')))
pooled = summarize(np.concatenate(list(loss_arrays.values())))
reference_pooled = summarize(np.concatenate(list(reference_arrays.values())))
print('All layers:', pooled)
print('True-AD forward reference:', reference_pooled)
print('True-AD mean by layer:', {k: v['true_ad']['mean'] for k,v in results.items()})
REPORT_DIR.mkdir(parents=True, exist_ok=True)
report = dict(encoder_checkpoint=str(ENCODER_CHECKPOINT), encoder_sha256=file_hash(ENCODER_CHECKPOINT),
    best_epoch=checkpoint['best_epoch'], decoder_checkpoint=str(LRN_AD_CHECKPOINT),
    decoder_sha256=signature['decoder_sha256'], seed=SEED, poisson_noise=True,
    target='clean expected counts', loss=signature['loss'], output_scale=OUTPUT_SCALE,
    tail_start=signature['tail_start'], tail_weight=signature['tail_weight'],
    selection='random without replacement per layer count; may overlap training',
    by_layers=results, pooled=pooled, true_ad_pooled=reference_pooled)
(REPORT_DIR/'reconstruction_statistics.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf8')
np.savez_compressed(REPORT_DIR/'reconstruction_losses.npz',
    **{f'encoder_{k}':v for k,v in loss_arrays.items()},
    **{f'true_ad_{k}':v for k,v in reference_arrays.items()},
    **{f'global_rows_{k}':v for k,v in selected_rows.items()})

## 4. Representative spectra: ten rows ? four columns

In [ ]:
fig, axes = plt.subplots(len(LAYER_COUNTS), 4, figsize=(22, 3*len(LAYER_COUNTS)), constrained_layout=True, squeeze=False)
for row, layers in enumerate(LAYER_COUNTS):
    preview = previews[layers]
    for col, percentile in enumerate((25,50,75,95)):
        ax = axes[row,col]
        ax.plot(preview['noisy'][col], color='0.7', linewidth=.6, label='Poisson input')
        ax.plot(preview['clean'][col], linewidth=1, label='Clean target')
        ax.plot(preview['reconstructed'][col], linewidth=1, label='Encoder + frozen LRN_AD')
        ax.set_title(f"{layers} layer(s), near p{percentile}; chi?={preview['losses'][col]:.4g}")
        ax.set_xlabel('Channel'); ax.set_ylabel('Counts')
        if row == 0 and col == 0: ax.legend(fontsize=8)
fig.savefig(REPORT_DIR/'representative_spectra.png', dpi=150)
plt.show()
print('Report and figure saved to:', REPORT_DIR)

In [ ]:
# Recompute only the displayed cases; rerun this cell without repeating the 100k evaluation.
# Convert elemental AD into SIMNRA thickness/fractions without thresholding or truncating layers.
SIMNRA_WORKERS = 4
simnra_methods = [ibl.SimnraMethod(m.label, Path(m.reference_file)) for m in study._config().methods]
case_order, experiments, predicted_ad_rows = [], [], []
with torch.inference_mode():
    for layers in LAYER_COUNTS:
        predicted_ad = model(torch.from_numpy(previews[layers]['noisy']).to(device)).cpu().numpy()
        predicted_ad = predicted_ad.reshape(-1, MAX_LAYERS, len(ELEMENTS)).astype(np.float64)/AD_FACTOR
        for col, ad in enumerate(predicted_ad):
            if not np.isfinite(ad).all() or (ad < 0).any():
                raise ValueError('Invalid encoder AD for SIMNRA')
            sample_layers = []
            for layer_index, row in enumerate(ad):
                template = study.experiment.sample.layers[layer_index]
                thickness = float(row.sum())
                # An exactly empty layer has arbitrary composition; retain the template in that case.
                composition = row/thickness if thickness > 0 else np.array([
                    next(s.concentration for s in template.species if s.element == e) for e in ELEMENTS])
                species = [ibl.Species(e, float(fraction),
                    isotopes=next(s.isotopes for s in template.species if s.element == e))
                    for e, fraction in zip(ELEMENTS, composition)]
                sample_layers.append(ibl.Layer(thickness, species, roughness=template.roughness,
                    porosity_fraction=template.porosity_fraction, pore_diameter=template.pore_diameter))
            experiments.append(ibl.Experiment(ibl.Sample(tuple(sample_layers)), study.experiment.detectors))
            case_order.append((layers, col))
            predicted_ad_rows.append(ad)
with ibl.SimnraSimulator(simnra_methods, workers=SIMNRA_WORKERS) as simulator:
    simulated = []
    for start in tqdm(range(0, len(experiments), SIMNRA_WORKERS), desc='SIMNRA representative cases'):
        simulated.extend(simulator.simulate_many(experiments[start:start+SIMNRA_WORKERS]))
simnra_report = []
for (layers, col), result in zip(case_order, simulated):
    if result.failed:
        raise RuntimeError(f'SIMNRA failed for {layers} layers, column {col+1}: {result.failure}')
    counts = np.asarray(result[METHOD].counts, dtype=np.float32)
    if len(counts) > WIDTH or not np.isfinite(counts).all() or (counts < 0).any():
        raise ValueError('Invalid SIMNRA spectrum/channel width')
    counts = np.pad(counts, (0, WIDTH-len(counts)))
    preview = previews[layers]
    if 'simnra' not in preview:
        preview['simnra'] = np.empty_like(preview['clean'])
        preview['simnra_losses'] = np.empty(len(preview['clean']))
    preview['simnra'][col] = counts
    with torch.inference_mode():
        score = float(per_sample_loss(torch.from_numpy(counts[None]).to(device)*OUTPUT_SCALE,
            torch.from_numpy(preview['clean'][col:col+1]).to(device))[0])
    preview['simnra_losses'][col] = score
    simnra_report.append(dict(layers=layers, column=col+1, global_row=int(preview['indices'][col]),
        surrogate_chi2=float(preview['losses'][col]), simnra_chi2=score))
REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR/'representative_simnra_statistics.json').write_text(
    json.dumps(dict(cases=simnra_report, mean_simnra_chi2=float(np.mean([r['simnra_chi2'] for r in simnra_report])),
        note='40 percentile-selected examples; not the mean over the full validation dataset'), indent=2)+'\n', encoding='utf8')
np.savez_compressed(REPORT_DIR/'representative_simnra_spectra.npz',
    physical_ad=np.stack(predicted_ad_rows), case_order=np.array(case_order),
    simnra_counts=np.stack([previews[k]['simnra'][j] for k,j in case_order]))
print(f"{'Layers':>6} {'Column':>6} {'LRN_AD chi2':>14} {'SIMNRA chi2':>14}")
for r in simnra_report:
    print(f"{r['layers']:6d} {r['column']:6d} {r['surrogate_chi2']:14.6g} {r['simnra_chi2']:14.6g}")


fig, axes = plt.subplots(len(LAYER_COUNTS), 4, figsize=(22, 3*len(LAYER_COUNTS)), constrained_layout=True, squeeze=False)
for row, layers in enumerate(LAYER_COUNTS):
    preview = previews[layers]
    for col, percentile in enumerate((25,50,75,95)):
        ax = axes[row,col]
        ax.plot(preview['noisy'][col], color='0.7', linewidth=.6, label='Poisson input')
        ax.plot(preview['clean'][col], linewidth=1, label='Clean target')
        ax.plot(preview['reconstructed'][col], linewidth=1, label='Encoder + frozen LRN_AD')
        ax.plot(preview['simnra'][col], linewidth=1, linestyle='--', label='Encoder + SIMNRA')
        ax.set_title(f"{layers} layer(s), near p{percentile}; LRN={preview['losses'][col]:.4g}, SIMNRA={preview['simnra_losses'][col]:.4g}")
        ax.set_xlabel('Channel'); ax.set_ylabel('Counts')
        if row == 0 and col == 0: ax.legend(fontsize=8)
fig.savefig(REPORT_DIR/'representative_spectra_simnra.png', dpi=150)
plt.show()
print('Report and figure saved to:', REPORT_DIR)

In [ ]:


# New independent pileup experiment: 10 new samples for each of 1?10 layers.
from dataclasses import replace
import tempfile
PILEUP_REFERENCE = ROOT/'xnra/Ref_RBS_LiCOFNaAlSiPSTiMnFeCoNiCuH.xnra'
PILEUP_SEED = SEED+10000
PILEUP_SAMPLES_PER_LAYER = 10
PILEUP_POISSON_NOISE = True
PILEUP_FUDGE_FACTOR_SECONDS = 0.4e-6
if not PILEUP_REFERENCE.is_file():
    raise FileNotFoundError(PILEUP_REFERENCE)
pileup_configuration = replace(configuration, mixed_samples=PILEUP_SAMPLES_PER_LAYER,
    pure_samples_per_element=0, seed=PILEUP_SEED)
pileup_methods = [ibl.SimnraMethod(METHOD, PILEUP_REFERENCE)]

def experiment_from_ad(ad, detectors):
    layers = []
    for i, row in enumerate(np.asarray(ad, dtype=np.float64).reshape(MAX_LAYERS, len(ELEMENTS))):
        if not np.isfinite(row).all() or (row < 0).any():
            raise ValueError('Invalid areal density')
        template = study.experiment.sample.layers[i]
        thickness = float(row.sum())
        fractions = row/thickness if thickness > 0 else np.array([
            next(s.concentration for s in template.species if s.element == e) for e in ELEMENTS])
        species = [ibl.Species(e, float(f), isotopes=next(s.isotopes for s in template.species if s.element == e))
                   for e,f in zip(ELEMENTS, fractions)]
        layers.append(ibl.Layer(thickness, species, roughness=template.roughness,
            porosity_fraction=template.porosity_fraction, pore_diameter=template.pore_diameter))
    return ibl.Experiment(ibl.Sample(tuple(layers)), detectors)

def align_simnra_spectra(results, minimum_channels=0):
    # SIMNRA varies returned extent with the sample. Channel calibration is fixed;
    # align channel zero and append zeros only, preserving every returned count.
    lengths = np.array([len(result[METHOD].counts) for result in results], dtype=np.int64)
    if len(lengths) == 0 or np.any(lengths <= 0):
        raise ValueError('SIMNRA returned an empty spectrum')
    width = max(int(lengths.max()), int(minimum_channels))
    if width % 2 == 0: width += 1
    aligned = np.zeros((len(results), width), np.float32)
    for row, result in enumerate(results):
        counts = np.asarray(result[METHOD].counts, dtype=np.float32)
        if not np.isfinite(counts).all() or (counts < 0).any():
            raise ValueError('SIMNRA returned invalid counts')
        aligned[row, :len(counts)] = counts
    return aligned, lengths

def remove_noisy_pileup(counts, detector, fudge_factor):
    # Noise-tolerant algebraic inverse of the native fast-pileup model.
    # Unlike the exact inverse, clip negative channel residuals and do not require
    # the noisy tail to match a deterministic convolution exactly. No clean target is used.
    observed = np.asarray(counts, dtype=np.float64)
    if observed.ndim != 1 or len(observed) % 2 != 1 or len(observed)//2+1 < WIDTH:
        raise ValueError('Pileup correction needs the full odd-length 2*N-1 spectrum covering the encoder grid')
    if detector.real_time <= 0 or detector.live_time <= 0:
        raise ValueError('Pileup correction needs positive real/live times')
    if not np.isfinite(observed).all() or (observed < 0).any():
        raise ValueError('Invalid noisy pileup counts')
    observed = observed/(detector.live_time/detector.real_time)
    total = float(observed.sum())
    size = len(observed)//2+1
    if total == 0:
        return np.zeros(size, np.float32), 0
    def fraction(count):
        u = fudge_factor*count/detector.real_time
        return u*np.exp(-u) if u < 745 else 0.0
    lower, upper = 1.0, 1/(1-np.exp(-1))
    for _ in range(100):
        ratio = (lower+upper)/2
        if ratio*(1-fraction(total*ratio)) < 1: lower = ratio
        else: upper = ratio
    original_total = total*(lower+upper)/2
    h = fraction(original_total)
    linear = 1-2*h
    observed = observed/original_total
    probabilities = np.zeros(size, np.float64)
    probabilities[0] = 2*observed[0]/(linear+np.sqrt(linear*linear+4*h*observed[0]))
    denominator = linear+2*h*probabilities[0]
    clipped = 0
    for channel in range(1, size):
        interior = np.dot(probabilities[1:channel], probabilities[channel-1:0:-1])
        residual = observed[channel]-h*interior
        clipped += int(residual < 0)
        probabilities[channel] = max(0.0, residual)/denominator
    return (probabilities*original_total).astype(np.float32), clipped

pileup_true_experiments, pileup_true_ad, pileup_order = [], [], []
for layer_count in range(1, 11):
    case = pileup_configuration.study(layer_count)
    parameters, _ = pileup_configuration.rows(layer_count)
    for j, row in enumerate(parameters):
        ad = true_density(row, case.parameter_names, case).reshape(MAX_LAYERS, len(ELEMENTS))/AD_FACTOR
        # Only real layers for generation; retain all encoder output layers for reconstruction.
        full = experiment_from_ad(ad, case.experiment.detectors)
        pileup_true_experiments.append(ibl.Experiment(ibl.Sample(full.sample.layers[:layer_count]), full.detectors))
        pileup_true_ad.append(ad)
        pileup_order.append((layer_count, j+1))

# Export the loaded checkpoint to a temporary package so standard native preprocessing
# corrects calibration/exposure after noise-tolerant pileup removal before attention inference.
# The pre-existing ZIP may contain an earlier epoch, so it is deliberately not used.
with tempfile.TemporaryDirectory(prefix='encoder-pileup-validation-') as temporary:
    package = Path(temporary)/'encoder_attention.zip'
    cpu_encoder = SpectrumAttentionEncoder(WIDTH, signature['output_size'], **signature['architecture']).eval()
    cpu_encoder.load_state_dict(checkpoint['model_state_dict'])
    cpu_encoder.export(package, study=study, method=METHOD, elements=ELEMENTS,
        ad_input_factor=AD_FACTOR, need_pileup_subtraction=True,
        pileup_fudge_factor_seconds=PILEUP_FUDGE_FACTOR_SECONDS)
    inverse = ibl.InverseModel(package, threads=8)
    with ibl.SimnraSimulator(pileup_methods, workers=SIMNRA_WORKERS) as simulator:
        generated = []
        for start in tqdm(range(0, len(pileup_true_experiments), SIMNRA_WORKERS), desc='Generate 100 pileup spectra'):
            generated.extend(simulator.simulate_many(pileup_true_experiments[start:start+SIMNRA_WORKERS]))
        if any(r.failed for r in generated):
            raise RuntimeError('SIMNRA pileup generation failed')
        pileup_clean, pileup_generated_lengths = align_simnra_spectra(generated, 2*WIDTH-1)
        print(f'SIMNRA generated lengths: {pileup_generated_lengths.min()}?{pileup_generated_lengths.max()}; aligned to {pileup_clean.shape[1]} channels')
        if not np.isfinite(pileup_clean).all() or (pileup_clean < 0).any():
            raise ValueError('Invalid generated pileup counts')
        rng = np.random.default_rng(PILEUP_SEED+1)
        pileup_noisy = rng.poisson(pileup_clean).astype(np.float32) if PILEUP_POISSON_NOISE else pileup_clean.copy()
        # Apply Poisson noise BEFORE correction: these are simulated measured spectra.
        corrected = [remove_noisy_pileup(counts, experiment.detectors[0], PILEUP_FUDGE_FACTOR_SECONDS)
                     for counts, experiment in zip(pileup_noisy, pileup_true_experiments)]
        pileup_corrected = np.stack([item[0] for item in corrected])
        pileup_clipped_channels = np.array([item[1] for item in corrected])
        inverse_inputs = [ibl.InverseInput({METHOD: ibl.Spectrum(METHOD, counts)}, experiment.detectors,
                                           pileup_already_removed=True)
                          for counts, experiment in zip(pileup_corrected, pileup_true_experiments)]
        pileup_predictions = inverse.predict(inverse_inputs)
        pileup_predicted_ad = np.stack([p.values.reshape(MAX_LAYERS, len(ELEMENTS)) for p in pileup_predictions])
        predicted_experiments = [experiment_from_ad(ad, original.detectors)
                                for ad, original in zip(pileup_predicted_ad, pileup_true_experiments)]
        reconstructed = []
        for start in tqdm(range(0, len(predicted_experiments), SIMNRA_WORKERS), desc='Reconstruct 100 pileup spectra'):
            reconstructed.extend(simulator.simulate_many(predicted_experiments[start:start+SIMNRA_WORKERS]))
        if any(r.failed for r in reconstructed):
            raise RuntimeError('SIMNRA pileup reconstruction failed')
        pileup_reconstructed, pileup_reconstructed_lengths = align_simnra_spectra(reconstructed, pileup_clean.shape[1])
        # Reconstruction can extend beyond generation; keep its tail in the comparison.
        extra = pileup_reconstructed.shape[1]-pileup_clean.shape[1]
        if extra:
            pileup_clean = np.pad(pileup_clean, ((0,0),(0,extra)))
            pileup_noisy = np.pad(pileup_noisy, ((0,0),(0,extra)))
if pileup_reconstructed.shape != pileup_clean.shape or not np.isfinite(pileup_reconstructed).all():
    raise ValueError('Invalid SIMNRA reconstruction dimensions/counts')
# Evaluate the same scaled chi2 formula, now in the original pileup measurement space.
measurement_weights = np.ones(pileup_clean.shape[1], np.float64)
measurement_weights[signature['tail_start']:] = signature['tail_weight']
measurement_weights /= measurement_weights.mean()
target = pileup_clean.astype(np.float64)*OUTPUT_SCALE
pileup_losses = (measurement_weights*(pileup_reconstructed.astype(np.float64)*OUTPUT_SCALE-target)**2/(target+1)).mean(1)
pileup_statistics = {layers: summarize(pileup_losses[(layers-1)*10:layers*10]) for layers in range(1,11)}
print('New pileup cases: encoder + SIMNRA chi2 against expected pileup spectra')
for layers, stats in pileup_statistics.items():
    print(f"{layers:2d} layers: mean={stats['mean']:.6g}; median={stats['median']:.6g}; p95={stats['p95']:.6g}")
(REPORT_DIR/'new_pileup_statistics.json').write_text(json.dumps(dict(seed=PILEUP_SEED,
    samples=100, poisson_noise=PILEUP_POISSON_NOISE, reference=str(PILEUP_REFERENCE),
    reference_sha256=file_hash(PILEUP_REFERENCE), encoder_best_epoch=checkpoint['best_epoch'],
    pileup_fudge_factor_seconds=PILEUP_FUDGE_FACTOR_SECONDS,
    preprocessing='noise-tolerant algebraic pileup removal; negative residuals clipped',
    clipped_channels_per_case=pileup_clipped_channels.tolist(),
    comparison_channels=int(pileup_clean.shape[1]),
    generated_channel_lengths=pileup_generated_lengths.tolist(),
    reconstructed_channel_lengths=pileup_reconstructed_lengths.tolist(),
    metric='scaled tail-weighted chi2 against clean pileup spectrum; original measurement space',
    by_layers=pileup_statistics, pooled=summarize(pileup_losses)), indent=2)+'\n', encoding='utf8')
np.savez_compressed(REPORT_DIR/'new_pileup_spectra.npz', case_order=np.array(pileup_order),
    expected=pileup_clean, noisy=pileup_noisy, corrected=pileup_corrected,
    clipped_channels=pileup_clipped_channels, reconstructed=pileup_reconstructed,
    generated_channel_lengths=pileup_generated_lengths, reconstructed_channel_lengths=pileup_reconstructed_lengths,
    true_ad=np.stack(pileup_true_ad), predicted_ad=pileup_predicted_ad, chi2=pileup_losses)
fig, axes = plt.subplots(10, 10, figsize=(40, 30), constrained_layout=True)
for i, (layers, example) in enumerate(pileup_order):
    ax = axes[layers-1, example-1]
    ax.plot(pileup_noisy[i], color='0.75', linewidth=.5, label='Poisson pileup input' if PILEUP_POISSON_NOISE else 'Pileup input')
    ax.plot(pileup_clean[i], linewidth=.8, label='Expected pileup spectrum')
    ax.plot(pileup_reconstructed[i], linewidth=.8, linestyle='--', label='Attention encoder + SIMNRA (pileup)')
    ax.set_title(f'{layers} layers, case {example}; chi2={pileup_losses[i]:.3g}', fontsize=8)
    ax.set_xlabel('Channel', fontsize=7); ax.set_ylabel('Counts', fontsize=7)
    ax.tick_params(labelsize=6)
    if i == 0: ax.legend(fontsize=6)
fig.savefig(REPORT_DIR/'new_pileup_spectra.png', dpi=150)
plt.show()
print('100 new pileup cases saved to:', REPORT_DIR)
